# MLflow Run Inventory & Pruning Helper

It supports two situations:

1. **Full tracking store**: you have both `mlflow.db` and an MLflow artifact store.  
2. **Artifact-only export**: you only have per-run `artifacts/` folders (common if you zipped `mlruns/` but not `mlflow.db`).

For each run, the notebook reports:
- last modified time (as a proxy for "age" when DB timestamps are unavailable),
- whether it looks complete (timings + baseline metrics + FACTER metrics),
- whether it contains stage error logs,
- artifact size and key artifact folders.

It then produces:
- `runs_inventory.csv` (all runs),
- `runs_candidates.csv` (runs you likely want to keep),
- `keep_run_ids.txt` (one run_id per line).

Nothing is deleted. Optionally, you can **copy** selected runs into a `pruned_mlruns/` folder for zipping.


In [ ]:
from __future__ import annotations

import os
import json
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_upwards(target_rel: Path) -> Path | None:
    """Search upwards from CWD for a relative path that exists."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        cand = (base / target_rel).resolve()
        if cand.exists():
            return cand
    return None


plt.rcParams["figure.figsize"] = (10, 5)

# CONFIG

ZIP_PATH = find_upwards(Path("mlruns.zip")) or Path("mlruns.zip")
BASE_DIR = ZIP_PATH.resolve().parent if ZIP_PATH.exists() else Path.cwd().resolve()

EXTRACT_DIR = BASE_DIR / "mlflow_prune_workspace/extracted"

DB_PATH_HINT = find_upwards("mlflow.db")  # e.g., Path("mlflow.db")

# Selection defaults
DAYS_BACK = 3                    # "recent" window
REQUIRE_FINISHED = True          # only possible if DB exists; otherwise uses artifact heuristics
REQUIRE_NO_STAGE_ERRORS = True
REQUIRE_TIMINGS = True

# If you want to keep only runs that completed the online loop, set True:
REQUIRE_FACTER_METRICS = True

OUTPUT_DIR = BASE_DIR / "mlflow_prune_workspace/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Config:")
print("  ZIP_PATH:", ZIP_PATH.resolve() if ZIP_PATH.exists() else ZIP_PATH)
print("  BASE_DIR:", BASE_DIR)
print("  EXTRACT_DIR:", EXTRACT_DIR)
print("  OUTPUT_DIR:", OUTPUT_DIR)
print("  DAYS_BACK:", DAYS_BACK)
print("  REQUIRE_FACTER_METRICS:", REQUIRE_FACTER_METRICS)


Config:
  ZIP_PATH: /home/ozzy/Desktop/facter-repr/mlruns.zip
  BASE_DIR: /home/ozzy/Desktop/facter-repr
  EXTRACT_DIR: /home/ozzy/Desktop/facter-repr/mlflow_prune_workspace/extracted
  OUTPUT_DIR: /home/ozzy/Desktop/facter-repr/mlflow_prune_workspace/output
  DAYS_BACK: 3
  REQUIRE_FACTER_METRICS: True


### Extract zip

In [ ]:
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR.resolve())
else:
    raise FileNotFoundError(f"ZIP_PATH not found: {ZIP_PATH.resolve()}")


Extracted to: /home/ozzy/Desktop/facter-repr/mlflow_prune_workspace/extracted


### Locate mlruns

In [ ]:
def find_first(root: Path, name: str) -> Path | None:
    matches = list(root.rglob(name))
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None

mlruns_dir = find_first(EXTRACT_DIR, "mlruns")
db_path = DB_PATH_HINT if DB_PATH_HINT else find_first(EXTRACT_DIR, "mlflow.db")

print("mlruns_dir:", mlruns_dir)
print("db_path:", db_path)

if mlruns_dir is None or not mlruns_dir.exists():
    raise FileNotFoundError("Could not find an 'mlruns' directory inside the extracted data.")


mlruns_dir: /home/ozzy/Desktop/facter-repr/mlflow_prune_workspace/extracted/mlruns
db_path: /home/ozzy/Desktop/facter-repr/mlflow.db


### Helpers + artifact-only scan

In [12]:
RUN_STATUS = {1:"SCHEDULED",2:"RUNNING",3:"FINISHED",4:"FAILED",5:"KILLED"}

def dir_size_mb(p: Path) -> float:
    total = 0
    for root, _, files in os.walk(p):
        for fn in files:
            fp = Path(root) / fn
            try:
                total += fp.stat().st_size
            except FileNotFoundError:
                pass
    return total / (1024 * 1024)

def newest_mtime(p: Path) -> datetime | None:
    mt = 0.0
    for root, _, files in os.walk(p):
        for fn in files:
            fp = Path(root) / fn
            try:
                mt = max(mt, fp.stat().st_mtime)
            except FileNotFoundError:
                pass
    return datetime.fromtimestamp(mt) if mt > 0 else None

def safe_json_load(p: Path):
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return None

def scan_artifact_run(run_dir: Path) -> dict:
    run_id = run_dir.name
    art_dir = run_dir / "artifacts"

    subdirs = sorted([d.name for d in art_dir.iterdir() if d.is_dir()]) if art_dir.exists() else []
    results_dir = art_dir / "results"

    timings_path = results_dir / "timings.json"
    has_timings = timings_path.exists()
    timings = safe_json_load(timings_path) if has_timings else None

    baseline_metrics = list(results_dir.glob("baseline_metrics_*.json")) if results_dir.exists() else []
    facter_metrics = list(results_dir.glob("facter_metrics_*.json")) if results_dir.exists() else []

    stage_errors = list((art_dir / "stage_errors").glob("*.txt")) if (art_dir / "stage_errors").exists() else []

    total_seconds = None
    if isinstance(timings, dict):
        total_seconds = timings.get("stage.TOTAL.seconds") or timings.get("TOTAL.seconds") or timings.get("TOTAL")

    return {
        "run_id": run_id,
        "has_artifacts": art_dir.exists(),
        "artifact_subdirs": ",".join(subdirs),
        "has_results_dir": results_dir.exists(),
        "has_timings": has_timings,
        "total_seconds": total_seconds,
        "n_baseline_metrics_files": len(baseline_metrics),
        "n_facter_metrics_files": len(facter_metrics),
        "has_stage_errors": len(stage_errors) > 0,
        "n_stage_errors": len(stage_errors),
        "artifact_size_mb": dir_size_mb(art_dir) if art_dir.exists() else 0.0,
        "last_modified": newest_mtime(run_dir),
    }

def scan_mlruns_artifact_only(mlruns_dir: Path) -> pd.DataFrame:
    # Two common layouts:
    # (A) standard file-store: mlruns/<exp_id>/<run_id>/...
    # (B) artifact-only: mlruns/<run_id>/artifacts/...
    rows = []

    # Detect layout B
    top = [d for d in mlruns_dir.iterdir() if d.is_dir() and (d / "artifacts").exists()]
    if top:
        for rd in top:
            rows.append(scan_artifact_run(rd))
        return pd.DataFrame(rows)

    # Layout A
    exp_dirs = [d for d in mlruns_dir.iterdir() if d.is_dir()]
    for exp in exp_dirs:
        run_dirs = [d for d in exp.iterdir() if d.is_dir() and (d / "artifacts").exists()]
        for rd in run_dirs:
            rec = scan_artifact_run(rd)
            rec["experiment_dir"] = exp.name
            rows.append(rec)

    return pd.DataFrame(rows)

runs_art = scan_mlruns_artifact_only(mlruns_dir)
print("Runs discovered:", len(runs_art))
display(runs_art.head(10))


Runs discovered: 64


,run_id,has_artifacts,artifact_subdirs,has_results_dir,has_timings,total_seconds,n_baseline_metrics_files,n_facter_metrics_files,has_stage_errors,n_stage_errors,artifact_size_mb,last_modified
0,a2697541d45b4c92ac417a0e609f4a06,True,"config,data,results",True,True,None,2,1,False,0,41.759577,2026-01-29 03:16:48.985124
1,c332cadb09e2479c94287be997a34ea5,True,"config,data,results",True,True,None,2,1,False,0,22.631164,2026-01-29 03:16:51.686049
2,1b39d684c32b4df2ac523e64f4099d0b,True,"config,stage_errors",False,False,None,0,0,True,1,0.000674,2026-01-29 03:16:51.818404
3,dbe8f143ba374f4aaa65621ebd63fbe3,True,"config,data,results,stage_errors",True,False,None,2,0,True,1,22.720167,2026-01-29 03:16:52.121037
4,49351a18cab1452b9c8b87aab848722a,True,"config,data,results",True,True,None,2,1,False,0,41.805271,2026-01-29 03:16:51.310059
5,249335d2f529406ba2c5f07ff7ec6636,True,config,False,False,None,0,0,False,0,0.000113,2026-01-29 03:16:51.325335
6,8d7c0b405eca491384b323b45f35682e,True,"config,data,results",True,True,None,2,1,False,0,38.765233,2026-01-29 03:16:50.860072
7,987a89dc8f2a418db897c23db401f2a4,True,"config,data,results",True,True,None,2,1,False,0,38.763516,2026-01-29 03:16:52.550025
8,bde284c7dc9e45f78fce858c2a94ec4a,True,"config,data,results",True,True,None,2,1,False,0,23.819818,2026-01-29 03:16:49.058122
9,d6b5d12327d04ad299299ba33af95a57,True,"config,data,results",True,True,None,2,1,False,0,39.961471,2026-01-29 03:16:49.274116


### If we have a db path, load run status/timestamps from sqlite

In [ ]:
def load_runs_from_sqlite(db_path: Path) -> pd.DataFrame:
    import sqlite3
    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()

    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = {r[0] for r in cur.fetchall()}
    if "runs" not in tables:
        conn.close()
        raise RuntimeError("sqlite DB does not contain a 'runs' table. Cannot load statuses.")

    cols = [r[1] for r in cur.execute("PRAGMA table_info(runs);").fetchall()]
    want = ["run_uuid", "experiment_id", "status", "start_time", "end_time", "artifact_uri"]
    avail = [c for c in want if c in cols]

    q = f"SELECT {', '.join(avail)} FROM runs"
    cur.execute(q)
    rows = cur.fetchall()
    conn.close()

    df = pd.DataFrame(rows, columns=avail)

    # Convert timestamps (ms since epoch) if numeric
    for c in ["start_time", "end_time"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], unit="ms", errors="coerce")

    # Normalize status to string robustly
    if "status" in df.columns:
        def to_status_str(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return ""
            # If already a string like "FAILED"/"FINISHED"
            if isinstance(x, str):
                return x.strip().upper()
            # If numeric code
            try:
                xi = int(x)
                return RUN_STATUS.get(xi, str(xi))
            except Exception:
                return str(x)

        df["status_str"] = df["status"].apply(to_status_str)

    return df

db_df = None
if db_path is not None and Path(db_path).exists():
    try:
        db_df = load_runs_from_sqlite(Path(db_path))
        print("Loaded DB runs:", len(db_df))
        display(db_df.head(10))
    except Exception as e:
        print("DB load failed:", e)
else:
    print("No mlflow.db found in the extracted data. Proceeding with artifact-only analysis.")


In [ ]:
# 5) Merge DB info (if present) + artifacts info

df = runs_art.copy()

if db_df is not None and not db_df.empty:
    df = df.merge(
        db_df.rename(columns={"run_uuid": "run_id"}),
        on="run_id",
        how="left",
        suffixes=("", "_db"),
    )
    df["time_ref"] = df["end_time"].fillna(df["start_time"]).fillna(df["last_modified"])
else:
    df["time_ref"] = df["last_modified"]

df["days_ago"] = (pd.Timestamp.now() - pd.to_datetime(df["time_ref"])).dt.total_seconds() / 86400.0

df["complete_baseline"] = (df["n_baseline_metrics_files"] > 0) & df["has_timings"] & (~df["has_stage_errors"])
df["complete_facter"] = (df["n_facter_metrics_files"] > 0) & df["has_timings"] & (~df["has_stage_errors"])

df["score"] = (
    2.0 * df["has_timings"].astype(float)
    + 2.0 * (df["n_baseline_metrics_files"] > 0).astype(float)
    + 2.0 * (df["n_facter_metrics_files"] > 0).astype(float)
    + 1.0 * df["has_results_dir"].astype(float)
    - 3.0 * df["has_stage_errors"].astype(float)
)

df = df.sort_values(["score", "time_ref"], ascending=[False, False]).reset_index(drop=True)

print("Inventory rows:", len(df))
display(df.head(20))


In [ ]:
# 6) Summary

summary = {
    "runs_total": int(len(df)),
    "has_results_dir": int(df["has_results_dir"].sum()),
    "has_timings": int(df["has_timings"].sum()),
    "has_stage_errors": int(df["has_stage_errors"].sum()),
    "complete_baseline": int(df["complete_baseline"].sum()),
    "complete_facter": int(df["complete_facter"].sum()),
}

print("Summary:", summary)

if "status_str" in df.columns:
    print("\nBy DB status:")
    display(df["status_str"].value_counts(dropna=False))
else:
    print("\n(DB status unavailable: zip appears to contain artifacts only.)")

print("\nTop artifact_subdirs patterns:")
display(df["artifact_subdirs"].value_counts().head(10))


In [ ]:
# 7) Build candidate set you likely want to KEEP

cutoff = DAYS_BACK
cand = df.copy()

cand = cand[cand["days_ago"] <= cutoff]

if REQUIRE_FINISHED and "status_str" in cand.columns:
    cand = cand[cand["status_str"] == "FINISHED"]

if REQUIRE_NO_STAGE_ERRORS:
    cand = cand[~cand["has_stage_errors"]]

if REQUIRE_TIMINGS:
    cand = cand[cand["has_timings"]]

if REQUIRE_FACTER_METRICS:
    cand = cand[cand["n_facter_metrics_files"] > 0]

cand = cand.sort_values(["score", "time_ref"], ascending=[False, False]).reset_index(drop=True)

print(f"Candidates kept (days_back={DAYS_BACK}):", len(cand))
display(cand.head(50))

inv_path = OUTPUT_DIR / "runs_inventory.csv"
cand_path = OUTPUT_DIR / "runs_candidates.csv"
keep_path = OUTPUT_DIR / "keep_run_ids.txt"

df.to_csv(inv_path, index=False)
cand.to_csv(cand_path, index=False)
keep_path.write_text("\n".join(cand["run_id"].tolist()), encoding="utf-8")

print("Wrote:")
print(" -", inv_path)
print(" -", cand_path)
print(" -", keep_path)


In [ ]:
# 8) Optional: copy selected runs into a new folder for easy zipping (no deletion)

COPY_PRUNED = False
PRUNED_DIR = OUTPUT_DIR / "pruned_mlruns"
PRUNED_DIR.mkdir(exist_ok=True)

if COPY_PRUNED:
    import shutil
    kept = set(cand["run_id"].tolist())
    copied = 0
    # artifact-only layout: mlruns/<run_id>/...
    for run_id in kept:
        src = mlruns_dir / run_id
        if src.exists():
            dst = PRUNED_DIR / run_id
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            copied += 1
    print(f"Copied {copied} runs to:", PRUNED_DIR)
else:
    print("COPY_PRUNED is False (no files copied). Set COPY_PRUNED=True if you want a pruned folder to zip.")
